# language detection

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import re

In [2]:
df = pd.read_csv('language.csv')
df

,Text,language
0,klement gottwaldi surnukeha palsameeriti ning ...,Estonian
1,sebes joseph pereira thomas på eng the jesuit...,Swedish
2,ถนนเจริญกรุง อักษรโรมัน thanon charoen krung เ...,Thai
3,விசாகப்பட்டினம் தமிழ்ச்சங்கத்தை இந்துப் பத்திர...,Tamil
4,de spons behoort tot het geslacht haliclona en...,Dutch
...,...,...
21995,hors du terrain les années et sont des année...,French
21996,ใน พศ หลักจากที่เสด็จประพาสแหลมมลายู ชวา อินเ...,Thai
21997,con motivo de la celebración del septuagésimoq...,Spanish
21998,年月，當時還只有歲的她在美國出道，以mai-k名義推出首張英文《baby i like》，由...,Chinese


In [8]:
print(f"data shape : {df.shape}")
print("-"*20)
print(f"{df.info()}")
print("-"*20)
print(f"nulll check : \n{df.isnull().sum()}")

data shape : (22000, 2)
--------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22000 entries, 0 to 21999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Text      22000 non-null  object
 1   language  22000 non-null  object
dtypes: object(2)
memory usage: 343.9+ KB
None
--------------------
nulll check : 
Text        0
language    0
dtype: int64


In [9]:
df['language'].value_counts()

language
Estonian      1000
Swedish       1000
Thai          1000
Tamil         1000
Dutch         1000
Japanese      1000
Turkish       1000
Latin         1000
Urdu          1000
Indonesian    1000
Portugese     1000
French        1000
Chinese       1000
Korean        1000
Hindi         1000
Spanish       1000
Pushto        1000
Persian       1000
Romanian      1000
Russian       1000
English       1000
Arabic        1000
Name: count, dtype: int64

In [ ]:
# text preprocessing
def clean_text(text):
    text = text.lower()                          # "Hello" → "hello"
    text = re.sub(r'[0-9]', '', text)            # remove numbers
    text = re.sub(r'\s+', ' ', text).strip()     # remove extra spaces
    return text

df['Text'] = df['Text'].apply(clean_text)

In [12]:
# split row text first (prevent data leack)
x = np.array(df['Text'])
y = np.array(df['language'])

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y  # stratify = equal class ratio in both splits
)
print(f"train data : {X_train} {y_train}")
print('----------------------')
print(f'test data : {X_test} {y_test}')

train data : ['nucleele cometelor se numără printre cele mai întunecate obiecte cunoscute din sistemul solar sonda giotto a constatat că nucleul cometei halley reflectă din lumina care cade pe el pe când asfaltul reflectă'
 'los primeros paulinos desembarcaron en buenos aires en fundando la obra cardenal ferrari que dedico una particular atención al mundo universitario una de las fundadoras fue la escritora y política democratacristiana angélica felisa fuselli en se fundó el instituto grafotecnico considerada la primera escuela de periodismo de américa latina[]\u200b siempre en el entorno de la obra se organizaron grupos católicos de boys scout y se estructuró la acción católica argentina[]\u200b'
 'la revolución francesa tuvo una vocación universalista que se desarrolló de forma extensa en el imperio napoleónico que más que una monarquía universal creó un sistema continental encontinental system'
 ...
 'организация разработки единых сопоставимых на международном уровне стандартов клас

In [13]:
# vectorize fit on train only 
vectorize = CountVectorizer(max_features=5000) # limit vocabular size
X_train_voc = vectorize.fit_transform(X_train) # learn vocab from train only
X_test_voc = vectorize.fit_transform(X_test) # apply same vocab for test

In [14]:
# train model
model = MultinomialNB()
model.fit(X_train_voc , y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [15]:
# evaluate model properly
y_pred = model.predict(X_test_voc)
print(f"\nAccuracy: {model.score(X_test_voc, y_test):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))      # precision, recall per language
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.3439

Classification Report:
              precision    recall  f1-score   support

      Arabic       0.33      0.66      0.44       200
     Chinese       0.01      0.03      0.01       200
       Dutch       0.00      0.00      0.00       200
     English       0.15      0.01      0.02       200
    Estonian       0.52      0.12      0.20       200
      French       0.10      0.03      0.05       200
       Hindi       1.00      0.96      0.98       200
  Indonesian       0.34      0.08      0.13       200
    Japanese       0.00      0.02      0.01       200
      Korean       1.00      0.92      0.96       200
       Latin       0.39      0.23      0.28       200
     Persian       0.50      0.47      0.48       200
   Portugese       0.00      0.00      0.00       200
      Pushto       0.95      0.78      0.86       200
    Romanian       0.00      0.00      0.00       200
     Russian       0.99      0.98      0.99       200
     Spanish       0.22      0.01      

In [17]:
joblib.dump(model, 'models/language_model.pkl')
joblib.dump(vectorize, 'models/vectorizer.pkl')
print("Model saved.")

Model saved.


# test model

In [21]:
model = joblib.load('models/language_model.pkl')
vectorizer = joblib.load('models/vectorizer.pkl')

user_input = input("Enter text: ")
user_input_clean = clean_text(user_input)
user_vec = vectorizer.transform([user_input_clean])  # transform, never fit again
result = model.predict(user_vec)
print(f"Detected language: {result[0]}")

Detected language: Arabic
